In [1]:
import pandas as pd
import sys

PROJECT_PATH = r"C:\Users\abhir\OneDrive\Desktop\customer-retention-intelligence"

if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

DATA_PATH = "../data/customer_churn_dataset.csv"

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")

Dataset shape: (5000, 13)


In [2]:
from src.analytics import (
    overall_metrics,
    churn_by_category,
    churn_by_numeric_group,
    chi_square_test,
    t_test_by_churn
)

In [3]:
metrics = overall_metrics(df)

metrics

{'total_customers': 5000,
 'churned_customers': 1896,
 'retained_customers': 3104,
 'churn_rate': 0.3792}

In [4]:
competitor_results = churn_by_category(
    df,
    "competitor_usage"
)

competitor_results

,competitor_usage,customers,churned,churn_rate
0,0,3723,1160,31.157669
1,1,1277,736,57.635082


In [5]:
competitor_test = chi_square_test(
    df,
    "competitor_usage",
    "churn"
)

competitor_test

{'chi_square': 282.0449160841044,
 'p_value': 2.6914057819506316e-63,
 'degrees_of_freedom': 1,
 'effect_size': 0.23750575407097166,
 'contingency_table': churn                0     1
 competitor_usage            
 0                 2563  1160
 1                  541   736}

In [6]:
usage_test = t_test_by_churn(
    df,
    "monthly_usage_hours"
)

usage_test

{'retained_mean': 34.877190721649484,
 'churned_mean': 29.622732067510547,
 't_statistic': 11.209575593357513,
 'p_value': 8.815939531748819e-29}

In [7]:
tenure_test = t_test_by_churn(
    df,
    "tenure_months"
)

tenure_test

{'retained_mean': 22.030927835051546,
 'churned_mean': 16.41350210970464,
 't_statistic': 15.146473511051772,
 'p_value': 1.2646529870681256e-50}

In [8]:
satisfaction_test = t_test_by_churn(
    df,
    "satisfaction_score"
)

satisfaction_test

{'retained_mean': 3.7493556701030926,
 'churned_mean': 2.955168776371308,
 't_statistic': 36.13941551673399,
 'p_value': 4.517247557444448e-246}

In [9]:
support_test = t_test_by_churn(
    df,
    "support_tickets"
)

support_test

{'retained_mean': 1.3933634020618557,
 'churned_mean': 2.5537974683544302,
 't_statistic': -30.676726338963928,
 'p_value': 1.9696428523191504e-181}

In [10]:
subscription_results = churn_by_category(
    df,
    "subscription_type"
)

subscription_results

,subscription_type,customers,churned,churn_rate
0,Basic,2276,821,36.072056
1,Premium,1031,438,42.483026
2,Standard,1693,637,37.625517


In [11]:
subscription_test = chi_square_test(
    df,
    "subscription_type",
    "churn"
)

subscription_test

{'chi_square': 12.482951506756393,
 'p_value': 0.001946980139170718,
 'degrees_of_freedom': 2,
 'effect_size': 0.04996589137953288,
 'contingency_table': churn                 0    1
 subscription_type           
 Basic              1455  821
 Premium             593  438
 Standard           1056  637}

In [12]:
usage_bins = [0, 10, 20, 30, 40, 50, 60]

usage_groups = churn_by_numeric_group(
    df,
    "monthly_usage_hours",
    usage_bins
)

usage_groups

,monthly_usage_hours_group,customers,churned,churn_rate
0,"(-0.001, 10.0]",257,130,50.583658
1,"(10.0, 20.0]",940,426,45.319149
2,"(20.0, 30.0]",1267,513,40.489345
3,"(30.0, 40.0]",1064,393,36.936090
4,"(40.0, 50.0]",722,246,34.072022
5,"(50.0, 60.0]",407,119,29.238329


In [13]:
premium_df = df[
    df["subscription_type"] == "Premium"
].copy()

premium_df.shape

(1031, 13)

In [14]:
premium_usage = churn_by_numeric_group(
    premium_df,
    "monthly_usage_hours",
    usage_bins
)

premium_usage

,monthly_usage_hours_group,customers,churned,churn_rate
0,"(10.0, 20.0]",24,12,50.000000
1,"(20.0, 30.0]",195,99,50.769231
2,"(30.0, 40.0]",270,131,48.518519
3,"(40.0, 50.0]",237,105,44.303797
4,"(50.0, 60.0]",164,58,35.365854


In [15]:
discount_test = t_test_by_churn(
    df,
    "discount"
)

discount_test

{'retained_mean': 0.06019652061855671,
 'churned_mean': 0.056962025316455694,
 't_statistic': 1.4752302839853206,
 'p_value': 0.14022720044535394}

In [16]:
fee_test = t_test_by_churn(
    df,
    "monthly_fee"
)

fee_test

{'retained_mean': 44.68299935567009,
 'churned_mean': 47.7020358649789,
 't_statistic': -3.520817541565883,
 'p_value': 0.0004352014456729211}

In [17]:
fee_by_subscription = (
    df.groupby("subscription_type")["monthly_fee"]
    .agg(
        customers="count",
        average_fee="mean",
        minimum_fee="min",
        maximum_fee="max"
    )
    .reset_index()
)

fee_by_subscription

,subscription_type,customers,average_fee,minimum_fee,maximum_fee
0,Basic,2276,20.021054,14.99,24.98
1,Premium,1031,95.405441,80.10,109.99
2,Standard,1693,50.329687,40.01,59.99


In [18]:
subscription_churn = (
    df.groupby(["subscription_type", "churn"])
    .size()
    .reset_index(name="customers")
)

subscription_churn

,subscription_type,churn,customers
0,Basic,0,1455
1,Basic,1,821
2,Premium,0,593
3,Premium,1,438
4,Standard,0,1056
5,Standard,1,637


In [19]:
fee_by_type_test = {}

for subscription in df["subscription_type"].unique():

    subset = df[
        df["subscription_type"] == subscription
    ]

    fee_by_type_test[subscription] = t_test_by_churn(
        subset,
        "monthly_fee"
    )

fee_by_type_test

{'Basic': {'retained_mean': 20.03466666666667,
  'churned_mean': 19.996930572472593,
  't_statistic': 0.29864911633166735,
  'p_value': 0.7652449641060927},
 'Premium': {'retained_mean': 95.2468465430017,
  'churned_mean': 95.6201598173516,
  't_statistic': -0.6871414407163221,
  'p_value': 0.4921615830810948},
 'Standard': {'retained_mean': 50.25019886363636,
  'churned_mean': 50.461459968602824,
  't_statistic': -0.7401829891145851,
  'p_value': 0.4593172487791032}}

In [20]:
numeric_columns = [
    "age",
    "monthly_fee",
    "tenure_months",
    "monthly_usage_hours",
    "support_tickets",
    "payment_failures",
    "discount",
    "satisfaction_score",
    "competitor_usage",
    "churn"
]

correlation_matrix = df[numeric_columns].corr()

correlation_matrix["churn"].sort_values(ascending=False)

churn                  1.000000
support_tickets        0.420681
payment_failures       0.257498
competitor_usage       0.237978
monthly_fee            0.050331
age                   -0.020014
discount              -0.020689
monthly_usage_hours   -0.150950
tenure_months         -0.199035
satisfaction_score    -0.461046
Name: churn, dtype: float64